# Initial Set Up

## Install dependencies

In [1]:
!pip -q install transformers datasets accelerate

## Load Dataset and Classifier

In [3]:
import torch
from datasets import load_dataset
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

# SST-2 dataset
dataset = load_dataset("nyu-mll/glue", "sst2")

# Fixed pretrained sentiment classifier
classifier = pipeline(
    "text-classification",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=device
)

print(dataset)
print("Device:", "GPU" if device == 0 else "CPU")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})
Device: GPU


## Verify Predictions

In [4]:
reviews = [
    "The movie was slow and predictable.",
    "The movie was engaging and beautifully written.",
    "The acting was terrible, but the soundtrack was excellent."
]

for review in reviews:
    prediction = classifier(review, truncation=True)[0]

    print("Review:", review)
    print("Prediction:", prediction["label"])
    print("Confidence:", round(prediction["score"], 4))
    print("-" * 60)

Review: The movie was slow and predictable.
Prediction: NEGATIVE
Confidence: 0.9998
------------------------------------------------------------
Review: The movie was engaging and beautifully written.
Prediction: POSITIVE
Confidence: 0.9999
------------------------------------------------------------
Review: The acting was terrible, but the soundtrack was excellent.
Prediction: POSITIVE
Confidence: 0.9998
------------------------------------------------------------


## Install dependenices for auxillary LLM

In [6]:
!pip -q install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.6 MB/s eta 0:00:00


## Load LLM (which will generate counterfactuals)

In [7]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

if not torch.cuda.is_available():
    raise RuntimeError(
        "Please enable a GPU in Colab before loading this model."
    )

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto"
)

llm.eval()

print("LLM loaded successfully!")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully!


## Test Counterfactutal Generation

In [8]:
def generate_counterfactual(
    review,
    target_label,
    max_new_tokens=100
):
    prompt = f"""Rewrite the review so that its sentiment is {target_label}.

            Preserve the original meaning and change as little text as possible.
            Return only the rewritten review.

            Original review: {review}
            Rewritten review:"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(llm.device)

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


review = "The movie was slow and predictable."

candidate = generate_counterfactual(
    review,
    target_label="positive"
)

print("Original:", review)
print("Counterfactual:", candidate)
print("Classifier result:", classifier(candidate)[0])

Original: The movie was slow and predictable.
Counterfactual: The movie was slow and predictable, but it had its moments.
Classifier result: {'label': 'POSITIVE', 'score': 0.9954278469085693}


## Define Measurement Functions

| Metric                 | How it is measured                                         |
| ---------------------- | ---------------------------------------------------------- |
| Validity               | Whether the classifier predicts the requested target label |
| Edit distance          | Word-level edits between the original and generated text   |
| Relative edit distance | Word-level edit distance divided by original word count    |
| Generation latency     | Time taken to generate the candidate                       |
| Token usage            | Prompt tokens and generated tokens                         |
| Classifier confidence  | Confidence in the predicted label                          |
| LLM calls              | One generation call per example                            |





In [9]:
import time
import torch
import pandas as pd
from datasets import load_dataset

def word_edit_distance(original, candidate):
    """Levenshtein edit distance measured in words."""
    a = original.lower().split()
    b = candidate.lower().split()

    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    for i in range(len(a) + 1):
        dp[i][0] = i
    for j in range(len(b) + 1):
        dp[0][j] = j

    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,       # deletion
                dp[i][j - 1] + 1,       # insertion
                dp[i - 1][j - 1] + cost # substitution
            )

    return dp[-1][-1]


def predict_sentiment(text):
    result = classifier(text, truncation=True)[0]
    return result["label"].upper(), result["score"]


# One-shot Generations

In [10]:
def generate_one_shot_measured(original, target_label):
    prompt = f"""Rewrite the review so its sentiment is {target_label}.
              Preserve its original meaning and change as little as possible.
              Return only the rewritten review.

              Original review: {original}
              Rewritten review:"""

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(llm.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency = time.perf_counter() - start

    generated_ids = output[0, prompt_tokens:]
    candidate = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return {
        "candidate": candidate,
        "prompt_tokens": prompt_tokens,
        "generated_tokens": len(generated_ids),
        "generation_seconds": latency
    }

## Run one-shot generation on 50 examples

In [11]:
N = 50
validation_data = dataset["validation"].select(range(N))

results = []

for i, example in enumerate(validation_data):
    original = example["sentence"]

    # Determine the source label using the fixed classifier.
    original_label, original_confidence = predict_sentiment(original)

    # Target the opposite label.
    target_label = (
        "POSITIVE" if original_label == "NEGATIVE"
        else "NEGATIVE"
    )

    # Exactly one generation attempt; no repair.
    generation = generate_one_shot_measured(
        original, target_label
    )

    candidate = generation["candidate"]

    # Verify the generated candidate.
    predicted_label, confidence = predict_sentiment(candidate)

    edit_distance = word_edit_distance(original, candidate)
    original_word_count = max(len(original.split()), 1)

    results.append({
        "id": i,
        "original": original,
        "original_label": original_label,
        "target_label": target_label,
        "candidate": candidate,
        "predicted_label": predicted_label,
        "valid": predicted_label == target_label,
        "confidence": confidence,
        "edit_distance_words": edit_distance,
        "relative_edit_distance": edit_distance / original_word_count,
        "generation_seconds": generation["generation_seconds"],
        "prompt_tokens": generation["prompt_tokens"],
        "generated_tokens": generation["generated_tokens"],
        "llm_calls": 1
    })

    if (i + 1) % 10 == 0:
        print(f"Completed {i + 1}/{N} examples")

df = pd.DataFrame(results)

print("\nFirst 5 results:")
display(df.head())

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Completed 10/50 examples
Completed 20/50 examples
Completed 30/50 examples
Completed 40/50 examples
Completed 50/50 examples

First 5 results:


,id,original,original_label,target_label,candidate,predicted_label,valid,confidence,edit_distance_words,relative_edit_distance,generation_seconds,prompt_tokens,generated_tokens,llm_calls
0,0,it 's a charming and often affecting journey .,POSITIVE,NEGATIVE,it's a charming and often affecting journey.,POSITIVE,False,0.999884,4,0.444444,1.232995,79,10,1
1,1,unflinchingly bleak and desperate,NEGATIVE,POSITIVE,unflinchingly bleak and desperate,NEGATIVE,False,0.998970,0,0.000000,1.528856,76,8,1
2,2,allows us to hope that nolan is poised to emba...,POSITIVE,NEGATIVE,denies us hope that Nolan is poised to embark ...,POSITIVE,False,0.994063,5,0.250000,3.967277,90,22,1
3,3,"the acting , costumes , music , cinematography...",POSITIVE,NEGATIVE,"the acting, costumes, music, cinematography, a...",NEGATIVE,True,0.999788,12,0.600000,3.049602,92,24,1
4,4,"it 's slow -- very , very slow .",NEGATIVE,POSITIVE,"it's slow -- very, very slow.",NEGATIVE,False,0.999689,6,0.666667,0.816967,79,10,1


## Calculate the baseline metrics

### Summary

In [12]:
summary = pd.DataFrame([{
    "Examples": len(df),
    "Validity (%)": 100 * df["valid"].mean(),
    "Mean edit distance (words)": df["edit_distance_words"].mean(),
    "Mean relative edit distance": df["relative_edit_distance"].mean(),
    "Mean generation time (s)": df["generation_seconds"].mean(),
    "Total generation time (s)": df["generation_seconds"].sum(),
    "Mean prompt tokens": df["prompt_tokens"].mean(),
    "Mean generated tokens": df["generated_tokens"].mean(),
    "Total LLM calls": df["llm_calls"].sum()
}])

display(summary.round(4))

,Examples,Validity (%),Mean edit distance (words),Mean relative edit distance,Mean generation time (s),Total generation time (s),Mean prompt tokens,Mean generated tokens,Total LLM calls
0,50,46.0,9.28,0.4922,2.12,106.0001,90.42,23.3,50


### Inspect failures

In [13]:
failures = df[~df["valid"]][
    [
        "original",
        "target_label",
        "candidate",
        "predicted_label",
        "edit_distance_words"
    ]
]

print(f"Failed counterfactuals: {len(failures)}/{len(df)}")
display(failures.head(10))

Failed counterfactuals: 27/50


,original,target_label,candidate,predicted_label,edit_distance_words
0,it 's a charming and often affecting journey .,NEGATIVE,it's a charming and often affecting journey.,POSITIVE,4
1,unflinchingly bleak and desperate,POSITIVE,unflinchingly bleak and desperate,NEGATIVE,0
2,allows us to hope that nolan is poised to emba...,NEGATIVE,denies us hope that Nolan is poised to embark ...,POSITIVE,5
4,"it 's slow -- very , very slow .",POSITIVE,"it's slow -- very, very slow.",NEGATIVE,6
5,although laced with humor and a few fanciful t...,NEGATIVE,although laced with humor and a few fanciful t...,POSITIVE,0
6,a sometimes tedious film .,POSITIVE,a somewhat tedious film.,NEGATIVE,3
8,you do n't have to know about music to appreci...,NEGATIVE,"Even if you don't know about music, you still ...",POSITIVE,13
9,"in exactly 89 minutes , most of which passed a...",POSITIVE,"In exactly 89 minutes, most of which felt like...",NEGATIVE,20
11,it takes a strange kind of laziness to waste t...,POSITIVE,It shows a strange kind of laziness to waste t...,NEGATIVE,11
12,... the film suffers from a lack of humor ( so...,POSITIVE,... the film lacks humor (something needed to ...,NEGATIVE,10


### Save the experiment results

In [14]:
df.to_csv("one_shot_results.csv", index=False)
summary.to_csv("one_shot_summary.csv", index=False)

print("Saved one_shot_results.csv and one_shot_summary.csv")

Saved one_shot_results.csv and one_shot_summary.csv


#### Added one more measurement (Mead edit distance between valid and failed only)

In [16]:
valid_df = df[df["valid"]]
failed_df = df[~df["valid"]]

print("Valid candidates:", len(valid_df))
print("Failed candidates:", len(failed_df))

if len(valid_df) > 0:
    print(
        "Mean edit distance (valid only):",
        round(valid_df["edit_distance_words"].mean(), 4)
    )

if len(failed_df) > 0:
    print(
        "Mean edit distance (failed only):",
        round(failed_df["edit_distance_words"].mean(), 4)
    )

Valid candidates: 23
Failed candidates: 27
Mean edit distance (valid only): 10.5652
Mean edit distance (failed only): 8.1852


# Multi-Candidate Selection

In [25]:
K = 5
TEMPERATURE = 0.9
TOP_P = 0.9
MAX_NEW_TOKENS = 100

def generate_multiple_candidates(original, target_label, k=K):
    prompt = f"""Rewrite the review so its sentiment is {target_label}.

            Preserve the original meaning and change as little as possible.
            Return only the rewritten review.

            Original review: {original}
            Rewritten review:"""

    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(llm.device)

    prompt_tokens = inputs["input_ids"].shape[1]

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            num_return_sequences=k,
            pad_token_id=tokenizer.eos_token_id
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    candidates = []

    for output in outputs:
        generated_ids = output[prompt_tokens:]
        text = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

        candidates.append({
            "candidate": text,
            "prompt_tokens": prompt_tokens,
            "generated_tokens": len(generated_ids)
        })

    return candidates, elapsed


print(f"Configured candidates per review: {K}")
print(f"Maximum candidate generations: {len(df) * K}")

Configured candidates per review: 5
Maximum candidate generations: 250


## Run the Multi-Candidate Selection on the same 50 reviews

In [26]:
candidate_records = []
selected_records = []

for _, row in df.iterrows():
    original = row["original"]
    target = row["target_label"]
    example_id = row["id"]

    candidates, batch_seconds = generate_multiple_candidates(
        original, target, k=K
    )

    example_candidates = []

    for j, item in enumerate(candidates):
        text = item["candidate"]

        pred_label, confidence = predict_sentiment(text)
        valid = pred_label == target
        distance = word_edit_distance(original, text)

        # For this binary classifier, estimate target-label
        # confidence from the predicted class and its score.
        target_confidence = (
            confidence if valid else 1.0 - confidence
        )

        record = {
            "id": example_id,
            "candidate_id": j + 1,
            "original": original,
            "target_label": target,
            "candidate": text,
            "predicted_label": pred_label,
            "valid": valid,
            "confidence": confidence,
            "target_confidence": target_confidence,
            "edit_distance_words": distance,
            "relative_edit_distance": (
                distance / max(len(original.split()), 1)
            ),
            "prompt_tokens": item["prompt_tokens"],
            "generated_tokens": item["generated_tokens"],
            "generation_batch_seconds": batch_seconds
        }

        candidate_records.append(record)
        example_candidates.append(record)

    # Prefer the valid candidate with minimum edit distance.
    valid_candidates = [
        c for c in example_candidates if c["valid"]
    ]

    if valid_candidates:
        selected = min(
            valid_candidates,
            key=lambda c: c["edit_distance_words"]
        )
        selection_reason = "minimum edit distance among valid candidates"
    else:
        # Diagnostic fallback only: the example remains a failure.
        selected = max(
            example_candidates,
            key=lambda c: c["target_confidence"]
        )
        selection_reason = "no valid candidate; best target confidence"

    selected_records.append({
        **selected,
        "selection_reason": selection_reason,
        "candidates_generated": K,
        "valid_candidates": len(valid_candidates),
        "example_success": len(valid_candidates) > 0,
        "llm_generation_batches": 1,
        "candidate_generation_attempts": K
    })

    if len(selected_records) % 10 == 0:
        print(f"Completed {len(selected_records)}/{len(df)} examples")

candidates_df = pd.DataFrame(candidate_records)
multi_df = pd.DataFrame(selected_records)

print("Candidate generations:", len(candidates_df))
print("Examples evaluated:", len(multi_df))

Completed 10/50 examples
Completed 20/50 examples
Completed 30/50 examples
Completed 40/50 examples
Completed 50/50 examples
Candidate generations: 250
Examples evaluated: 50


## Calulate the baseline metrics

### Summary

In [31]:
summary_multi = pd.DataFrame([{
    "Examples": len(multi_df),
    "Candidates per example": K,
    "Total candidates generated": len(candidates_df),
    "Validity (%)": 100 * multi_df["example_success"].mean(),
    "Candidate validity (%)": 100 * candidates_df["valid"].mean(),
    "Mean edit distance (selected)": (
        multi_df["edit_distance_words"].mean()
    ),
    "Mean edit distance (valid selected)": (
        multi_df.loc[
            multi_df["example_success"], "edit_distance_words"
        ].mean()
    ),
    "Mean relative edit distance (selected)": (
        multi_df["relative_edit_distance"].mean()
    ),
    "Mean valid candidates per example": (
        multi_df["valid_candidates"].mean()
    ),
    "Total generation time (s)": (
        multi_df["generation_batch_seconds"].sum()
    ),
    "Mean generation time per example (s)": (
        multi_df["generation_batch_seconds"].mean()
    ),
    "Mean prompt tokens per candidate": (
        candidates_df["prompt_tokens"].mean()
    ),
    "Mean generated tokens per candidate": (
        candidates_df["generated_tokens"].mean()
    ),
    "Total generated tokens": (
        candidates_df["generated_tokens"].sum()
    ),
    "Generation batches": len(multi_df),
    "Candidate generation attempts": len(candidates_df)
}])

display(summary_multi.round(4))

,Examples,Candidates per example,Total candidates generated,Validity (%),Candidate validity (%),Mean edit distance (selected),Mean edit distance (valid selected),Mean relative edit distance (selected),Mean valid candidates per example,Total generation time (s),Mean generation time per example (s),Mean prompt tokens per candidate,Mean generated tokens per candidate,Total generated tokens,Generation batches,Candidate generation attempts
0,50,5,250,52.0,36.0,6.5,7.5,0.3549,1.8,122.6828,2.4537,90.42,23.28,5820,50,250


### Inspect failures

In [32]:
print("Examples with no valid candidate:",
      int((~multi_df["example_success"]).sum()))

display(
    multi_df.loc[
        ~multi_df["example_success"],
        ["original", "target_label", "candidate",
         "predicted_label", "target_confidence"]
    ].head(10)
)

Examples with no valid candidate: 24


,original,target_label,candidate,predicted_label,target_confidence
0,it 's a charming and often affecting journey .,NEGATIVE,it's a charming and often affecting journey.,POSITIVE,0.000116
1,unflinchingly bleak and desperate,POSITIVE,unflinchingly bleak and desperate,NEGATIVE,0.001030
4,"it 's slow -- very , very slow .",POSITIVE,"it's slow -- very, very slow.",NEGATIVE,0.000311
5,although laced with humor and a few fanciful t...,NEGATIVE,although laced with humor and a few fanciful t...,POSITIVE,0.000171
6,a sometimes tedious film .,POSITIVE,a sometimes tedious film.,NEGATIVE,0.000495
7,or doing last year 's taxes with your ex-wife .,POSITIVE,Doing last year's taxes with your ex-wife.,NEGATIVE,0.022815
8,you do n't have to know about music to appreci...,NEGATIVE,you have to know about music to appreciate the...,POSITIVE,0.000185
9,"in exactly 89 minutes , most of which passed a...",POSITIVE,"In exactly 89 minutes, most of which passed as...",NEGATIVE,0.000749
11,it takes a strange kind of laziness to waste t...,POSITIVE,It takes a strange kind of laziness to waste t...,NEGATIVE,0.000298
12,... the film suffers from a lack of humor ( so...,POSITIVE,... the film lacks humor (something needed to ...,NEGATIVE,0.000195


### Save the experiment results

In [33]:
candidates_df.to_csv("multi_candidate_all_results.csv", index=False)
multi_df.to_csv("multi_candidate_selected_results.csv", index=False)
summary_multi.to_csv("multi_candidate_summary.csv", index=False)

print("Saved all candidate, selected candidate, and summary results.")

Saved all candidate, selected candidate, and summary results.


#